# rank-world-size-args — worked example 2: Scatter protocol: src distributes chunks to each rank

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank-world-size-args`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a scatter operation, one source rank (`src`) sends a distinct chunk of data to each rank (including itself conceptually, but in practice it just keeps its own chunk). All non-source ranks receive exactly one chunk from `src`. The function signature `scatter_protocol(tensor, rank, world_size, src=0)` is the standard distributed primitive convention, with `rank` and `world_size` always explicitly passed.

## Worked solution

**Step 1 — Signature.** `scatter_protocol(tensor, rank, world_size, src=0)` follows the convention. The `src` parameter defaults to 0 (the master rank) but can be overridden.

**Step 2 — Source rank sends to each other.** If `rank == src`, the source sends a chunk to every other rank in ascending order. It does not 'send to itself' — the source keeps its own chunk.

**Step 3 — Non-source ranks receive.** If `rank != src`, the rank receives exactly one chunk from `src`.

**Step 4 — Verify.** We run the protocol for all ranks in a world of size 4 and confirm the total number of sends equals `world_size - 1` (one per non-source rank).

In [ ]:
import torch as t

def scatter_protocol(tensor, rank: int, world_size: int, src: int = 0) -> list:
    """Return (action, other_rank) tuples for scatter with given src."""
    if rank == src:
        return [('send', other) for other in range(world_size) if other != src]
    return [('recv', src)]

# Verify with world_size=4, default src=0
world_size = 4
for rank in range(world_size):
    actions = scatter_protocol(None, rank, world_size, src=0)
    print(f'rank={rank}: {actions}')

# Check total send count from src=0
src_actions = scatter_protocol(None, 0, world_size)
send_count = sum(1 for a in src_actions if a[0] == 'send')
print(f'\nSrc sends to {send_count} ranks (expect {world_size - 1})')  # 3